# 🏎️ Formula 1 World Championship (1950–2020)
## Course 602 – Data Analytics using Python
### Exploratory Data Analysis, Lap Time Prediction, Pit Stop Strategy & Podium Classification

**Dataset:** https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020  
**Student Name(s):**  
**Date:**  

---

## Setup: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures, StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report, roc_auc_score
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('All libraries loaded successfully')

All libraries loaded successfully


---
## Task 1: Data Loading and Understanding

> **Instructions:** Update `DATA_PATH` to the folder where you extracted the Kaggle dataset.

In [2]:
DATA_PATH = './csv/'

# Load all CSV files
races         = pd.read_csv(DATA_PATH + 'races.csv')
results       = pd.read_csv(DATA_PATH + 'results.csv')
lap_times     = pd.read_csv(DATA_PATH + 'lap_times.csv')
pit_stops     = pd.read_csv(DATA_PATH + 'pit_stops.csv')
drivers       = pd.read_csv(DATA_PATH + 'drivers.csv')
constructors  = pd.read_csv(DATA_PATH + 'constructors.csv')
circuits      = pd.read_csv(DATA_PATH + 'circuits.csv')
qualifying    = pd.read_csv(DATA_PATH + 'qualifying.csv')
status        = pd.read_csv(DATA_PATH + 'status.csv')

print('Files loaded:')
for name, df in [('races', races), ('results', results), ('lap_times', lap_times),
                 ('pit_stops', pit_stops), ('drivers', drivers), ('constructors', constructors),
                 ('circuits', circuits), ('qualifying', qualifying), ('status', status)]:
    print(f'  {name:20s} → {df.shape[0]:>7,} rows × {df.shape[1]:>3} cols')

Files loaded:
  races                →   1,125 rows ×  18 cols
  results              →  26,759 rows ×  18 cols
  lap_times            → 589,081 rows ×   6 cols
  pit_stops            →  11,371 rows ×   7 cols
  drivers              →     861 rows ×   9 cols
  constructors         →     212 rows ×   5 cols
  circuits             →      77 rows ×   9 cols
  qualifying           →  10,494 rows ×   9 cols
  status               →     139 rows ×   2 cols


In [3]:
# ── 1A: Explore the results table ───────────────────────────────────────────
print('=== results.csv ===' )
print('Shape:', results.shape)
print('\nFirst 5 rows:')
display(results.head())
print('\nLast 5 rows:')
display(results.tail())
print('\nData types:')
display(results.dtypes)
print('\nDescriptive statistics:')
display(results.describe())

=== results.csv ===
Shape: (26759, 18)

First 5 rows:


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1



Last 5 rows:


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
26754,26760,1144,825,210,20,14,16,16,16,0.0,57,\N,\N,57,1,1:25.637,222.002,11
26755,26761,1144,859,215,30,12,17,17,17,0.0,55,\N,\N,52,12,1:28.751,214.212,5
26756,26762,1144,822,15,77,9,\N,R,18,0.0,30,\N,\N,14,19,1:29.482,212.462,130
26757,26763,1144,861,3,43,20,\N,R,19,0.0,26,\N,\N,5,17,1:29.411,212.631,5
26758,26764,1144,815,9,11,10,\N,R,20,0.0,0,\N,\N,\N,0,\N,\N,4



Data types:


resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number                 str
grid                 int64
position               str
positionText           str
positionOrder        int64
points             float64
laps                 int64
time                   str
milliseconds           str
fastestLap             str
rank                   str
fastestLapTime         str
fastestLapSpeed        str
statusId             int64
dtype: object


Descriptive statistics:


,resultId,raceId,driverId,constructorId,grid,positionOrder,points,laps,statusId
count,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000
mean,13380.977391,551.687283,278.673530,50.180537,11.134796,12.794051,1.987632,46.301768,17.224971
std,7726.134642,313.265036,282.703039,61.551498,7.202860,7.665951,4.351209,29.496557,26.026104
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,6690.500000,300.000000,57.000000,6.000000,5.000000,6.000000,0.000000,23.000000,1.000000
50%,13380.000000,531.000000,172.000000,25.000000,11.000000,12.000000,0.000000,53.000000,10.000000
75%,20069.500000,811.000000,399.500000,63.000000,17.000000,18.000000,2.000000,66.000000,14.000000
max,26764.000000,1144.000000,862.000000,215.000000,34.000000,39.000000,50.000000,200.000000,141.000000


In [ ]:
# ── 1B: Build the Race Results Master DataFrame ──────────────────────────────
race_master = (
    results
    .merge(races[['raceId','year','round','circuitId','name']], on='raceId', how='left')
    .merge(drivers[['driverId','forename','surname','nationality']], on='driverId', how='left')
    .merge(constructors[['constructorId','name']], on='constructorId', how='left', suffixes=('_race','_constructor'))
    .merge(status[['statusId','status']], on='statusId', how='left')
)

race_master['driver_name'] = race_master['forename'] + ' ' + race_master['surname']
race_master['positionOrder'] = pd.to_numeric(race_master['positionOrder'], errors='coerce')
race_master['grid'] = pd.to_numeric(race_master['grid'], errors='coerce')
race_master['points'] = pd.to_numeric(race_master['points'], errors='coerce')

print('Race Master shape:', race_master.shape)
display(race_master.head(3))

Race Master shape: (26759, 28)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,year,round,circuitId,name_race,forename,surname,nationality,name_constructor,status,driver_name
0,1,18,1,1,22,1,1,1,1,10.0,...,2008,1,1,Australian Grand Prix,Lewis,Hamilton,British,McLaren,Finished,Lewis Hamilton
1,2,18,2,2,3,5,2,2,2,8.0,...,2008,1,1,Australian Grand Prix,Nick,Heidfeld,German,BMW Sauber,Finished,Nick Heidfeld
2,3,18,3,3,7,7,3,3,3,6.0,...,2008,1,1,Australian Grand Prix,Nico,Rosberg,German,Williams,Finished,Nico Rosberg


In [ ]:
# ── 1C: Build the Lap Times Master DataFrame (2010–2020 for performance) ──────
recent_races = races[races['year'] >= 2010]['raceId']
lap_master = (
    lap_times[lap_times['raceId'].isin(recent_races)]
    .merge(races[['raceId','year','round','circuitId']], on='raceId', how='left')
    .merge(results[['raceId','driverId','constructorId','grid']], on=['raceId','driverId'], how='left')
    .merge(drivers[['driverId','forename','surname']], on='driverId', how='left')
    .merge(constructors[['constructorId','name']], on='constructorId', how='left')
)
lap_master['driver_name'] = lap_master['forename'] + ' ' + lap_master['surname']

# Remove corrupt lap times (0 or null)
lap_master = lap_master[lap_master['milliseconds'] > 0].dropna(subset=['milliseconds'])

print('Lap Master shape:', lap_master.shape)
display(lap_master.head(3))

In [ ]:
# ── 1D: Identify data types in race_master ────────────────────────────────────
print('Quantitative – Continuous:', ['points', 'milliseconds', 'fastestLapSpeed'])
print('Quantitative – Discrete  :', ['grid', 'positionOrder', 'laps', 'round'])
print('Qualitative  – Nominal   :', ['nationality', 'name_constructor', 'status'])
print('Qualitative  – Ordinal   :', ['positionOrder', 'round'])

---
## Task 2: Exploratory Data Analysis (EDA)

### 2A — Univariate Analysis

In [ ]:
# Helper: plot histogram + boxplot side by side
def plot_univariate(series, title, xlabel, bins=60, color='steelblue'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    series = series.dropna()
    axes[0].hist(series, bins=bins, color=color, edgecolor='white', alpha=0.85)
    axes[0].set_title(f'{title} – Histogram')
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel('Frequency')
    axes[1].boxplot(series, vert=False, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.6))
    axes[1].set_title(f'{title} – Boxplot')
    axes[1].set_xlabel(xlabel)
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}_univariate.png', dpi=120)
    plt.show()

# Lap time (ms)
plot_univariate(lap_master['milliseconds'], 'Lap Time', 'Milliseconds')

# Points per race
plot_univariate(race_master['points'], 'Race Points', 'Points')

# Grid (starting) position
plot_univariate(race_master['grid'], 'Grid Position', 'Grid')

In [ ]:
# Pit stop duration univariate
pit_stops['duration_sec'] = pd.to_numeric(pit_stops['duration'], errors='coerce')
# Filter realistic pit stop durations (2–120 seconds)
pit_clean = pit_stops[(pit_stops['duration_sec'] >= 2) & (pit_stops['duration_sec'] <= 120)]
plot_univariate(pit_clean['duration_sec'], 'Pit Stop Duration', 'Seconds')

### 2B — Bivariate Analysis

In [ ]:
# Grid Position vs Final Position
sample = race_master.dropna(subset=['grid','positionOrder']).sample(5000, random_state=42)

plt.figure(figsize=(8,6))
plt.scatter(sample['grid'], sample['positionOrder'], alpha=0.15, s=10, color='tomato')
plt.xlabel('Grid (Starting) Position')
plt.ylabel('Final Finishing Position')
plt.title('Grid Position vs Final Position')
m, b = np.polyfit(sample['grid'], sample['positionOrder'], 1)
plt.plot(sorted(sample['grid']), [m*x+b for x in sorted(sample['grid'])], 'k--', linewidth=2, label=f'Trend (slope={m:.2f})')
plt.legend()
plt.savefig('grid_vs_position.png', dpi=120)
plt.show()

corr = sample['grid'].corr(sample['positionOrder'])
print(f'Pearson correlation (Grid vs Final Position): {corr:.4f}')

In [ ]:
# Average lap time trend by year
avg_lap_by_year = lap_master.groupby('year')['milliseconds'].median().reset_index()

plt.figure(figsize=(14,5))
plt.plot(avg_lap_by_year['year'], avg_lap_by_year['milliseconds']/1000, marker='o', color='darkorange', linewidth=2)
plt.xlabel('Year')
plt.ylabel('Median Lap Time (seconds)')
plt.title('Median Lap Time Trend by Year (2010–2020)')
plt.grid(True, alpha=0.4)
plt.savefig('lap_time_trend.png', dpi=120)
plt.show()

In [ ]:
# Correlation matrix — race_master numerical columns
num_cols = ['grid', 'positionOrder', 'points', 'laps']
corr_mat = race_master[num_cols].dropna().corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix – Race Results')
plt.savefig('correlation_heatmap.png', dpi=120)
plt.show()

### 2C — Multivariate Analysis

In [ ]:
# Lap times by constructor (Top 8 constructors only for clarity)
top_constructors = lap_master['name'].value_counts().head(8).index
lap_top = lap_master[lap_master['name'].isin(top_constructors)].copy()
lap_top['lap_time_sec'] = lap_top['milliseconds'] / 1000

plt.figure(figsize=(14, 6))
sns.boxplot(data=lap_top, x='name', y='lap_time_sec', palette='Set2')
plt.xticks(rotation=30, ha='right')
plt.xlabel('Constructor')
plt.ylabel('Lap Time (seconds)')
plt.title('Lap Time Distribution by Constructor (2010–2020)')
plt.tight_layout()
plt.savefig('lap_time_by_constructor.png', dpi=120)
plt.show()

---
## Task 3: Handling Missing Data and Outliers

In [ ]:
# ── Missing value analysis ──────────────────────────────────────────────────
def analyse_missing(df, name='DataFrame'):
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
    report = report[report['Missing Count'] > 0].sort_values('Missing %', ascending=False)
    print(f'Missing values in {name}:')
    display(report)
    
    if not report.empty:
        plt.figure(figsize=(10, 4))
        report['Missing %'].plot(kind='bar', color='salmon', edgecolor='white')
        plt.title(f'Missing Data % – {name}')
        plt.ylabel('Missing %')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

analyse_missing(race_master, 'Race Master')

In [ ]:
# ── Handle missing values ───────────────────────────────────────────────────

# 1. grid = 0 often means pit lane start — replace with median
race_master['grid'] = race_master['grid'].replace(0, np.nan)
race_master['grid'].fillna(race_master['grid'].median(), inplace=True)

# 2. Create 'finished' binary column from position
race_master['finished'] = race_master['positionOrder'].notna().astype(int)

# 3. fastestLapSpeed — impute with median per circuit
race_master['fastestLapSpeed'] = pd.to_numeric(race_master['fastestLapSpeed'], errors='coerce')
race_master['fastestLapSpeed'].fillna(race_master['fastestLapSpeed'].median(), inplace=True)

print('Missing values after handling:')
print(race_master[['grid','positionOrder','fastestLapSpeed','finished']].isnull().sum())

In [ ]:
# ── Outlier detection using IQR ─────────────────────────────────────────────
def detect_outliers_iqr(series, label):
    series = series.dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f'{label}: Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}')
    print(f'  Bounds: [{lower:.1f}, {upper:.1f}]')
    print(f'  Outliers: {len(outliers)} ({len(outliers)/len(series)*100:.2f}%)')
    return lower, upper

detect_outliers_iqr(lap_master['milliseconds'], 'Lap Time (ms)')
detect_outliers_iqr(pit_clean['duration_sec'], 'Pit Stop Duration (sec)')

---
## Task 4: Spread and Distribution of Data

In [ ]:
def spread_analysis(series, label):
    s = series.dropna()
    print(f'=== {label} ===')
    print(f'  Mean:     {s.mean():>12.2f}')
    print(f'  Median:   {s.median():>12.2f}')
    print(f'  Std Dev:  {s.std():>12.2f}')
    print(f'  Skewness: {stats.skew(s):>12.4f}  {"(right-skewed)" if stats.skew(s)>0 else "(left-skewed)"}')
    print(f'  Kurtosis: {stats.kurtosis(s):>12.4f}  {"(heavy tails)" if stats.kurtosis(s)>0 else "(light tails)"}')
    percentiles = np.percentile(s, [25, 50, 75, 90, 99])
    print(f'  Percentiles (25,50,75,90,99): {percentiles.round(1)}')

    # Histogram with KDE
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(s, bins=60, density=True, color='steelblue', edgecolor='white', alpha=0.7)
    s.plot.kde(ax=axes[0], color='darkred', linewidth=2)
    axes[0].set_title(f'{label} – Histogram + KDE')
    axes[0].set_xlabel(label)
    stats.probplot(s, plot=axes[1])
    axes[1].set_title(f'{label} – Q-Q Plot')
    plt.tight_layout()
    plt.savefig(f'{label.replace(" ","_")}_distribution.png', dpi=120)
    plt.show()
    print()

spread_analysis(lap_master['milliseconds'], 'Lap Time (ms)')
spread_analysis(race_master['points'].dropna(), 'Race Points')

---
## Task 5: Automating EDA using Python
### Reusable EDA Functions

In [ ]:
def summarise_dataframe(df, name='DataFrame'):
    """Print shape, dtypes, missing %, and descriptive stats for a DataFrame."""
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(f'Shape: {df.shape}')
    print(f'\nColumn Summary:')
    summary = pd.DataFrame({
        'dtype': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_%': (df.isnull().sum() / len(df) * 100).round(2),
        'unique': df.nunique()
    })
    display(summary)
    print('\nDescriptive Stats:')
    display(df.describe())

def plot_bivariate(df, x_col, y_col, hue=None, title=None):
    """Scatter plot with optional colour grouping."""
    plt.figure(figsize=(9, 5))
    if hue:
        for group, gdf in df.groupby(hue):
            plt.scatter(gdf[x_col], gdf[y_col], alpha=0.4, s=12, label=group)
        plt.legend(title=hue, bbox_to_anchor=(1,1))
    else:
        plt.scatter(df[x_col], df[y_col], alpha=0.3, s=12, color='steelblue')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(title or f'{x_col} vs {y_col}')
    plt.tight_layout()
    plt.show()

def plot_correlation_heatmap(df, title='Correlation Heatmap'):
    """Seaborn heatmap of correlation matrix for numeric columns."""
    corr = df.select_dtypes(include=np.number).corr()
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True,
                linewidths=0.4, annot_kws={'size': 9})
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=120)
    plt.show()

# Demo
summarise_dataframe(lap_master.head(10000), 'Lap Master (sample 10k)')

---
## Task 6 & 7: Regression Analysis – Lap Time Prediction

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────
reg_df = lap_master[['milliseconds','lap','grid','circuitId','constructorId','year','driverId']].dropna().copy()

# Encode categorical columns
for col in ['circuitId','constructorId','driverId']:
    le = LabelEncoder()
    reg_df[col] = le.fit_transform(reg_df[col].astype(str))

TARGET = 'milliseconds'
FEATURES_SIMPLE = ['lap']
FEATURES_MULTI  = ['lap', 'grid', 'circuitId', 'constructorId', 'year']

# Covariance and correlation with target
print('Pearson correlations with Lap Time:')
display(reg_df[FEATURES_MULTI + [TARGET]].corr()[TARGET].sort_values().to_frame())

In [ ]:
# ── Train / Validation / Test Split ─────────────────────────────────────────
X = reg_df[FEATURES_MULTI]
y = reg_df[TARGET]

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val           = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42)

print(f'Train size     : {len(X_train):,}')
print(f'Validation size: {len(X_val):,}')
print(f'Test size      : {len(X_test):,}')

In [ ]:
# ── Model Evaluation Helper ──────────────────────────────────────────────────
def evaluate_regression(model, X_tr, y_tr, X_ts, y_ts, name):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_ts)
    mae  = mean_absolute_error(y_ts, preds)
    mse  = mean_squared_error(y_ts, preds)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_ts, preds)
    print(f'\n── {name} ──')
    print(f'  MAE  : {mae:>10,.1f} ms  ({mae/1000:.2f} sec)')
    print(f'  MSE  : {mse:>10,.0f}')
    print(f'  RMSE : {rmse:>10,.1f} ms  ({rmse/1000:.2f} sec)')
    print(f'  R²   : {r2:>10.4f}')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results_log = []

# (i) Simple Linear Regression (lap number only)
slr = LinearRegression()
results_log.append(evaluate_regression(
    slr, X_train[FEATURES_SIMPLE], y_train,
    X_test[FEATURES_SIMPLE], y_test, 'Simple Linear Regression'))

# (ii) Multiple Linear Regression
mlr = LinearRegression()
results_log.append(evaluate_regression(mlr, X_train, y_train, X_test, y_test, 'Multiple Linear Regression'))

# (iii) Ridge Regression
ridge = Ridge(alpha=10)
results_log.append(evaluate_regression(ridge, X_train, y_train, X_test, y_test, 'Ridge Regression'))

# (iv) Lasso Regression
lasso = Lasso(alpha=10)
results_log.append(evaluate_regression(lasso, X_train, y_train, X_test, y_test, 'Lasso Regression'))

In [ ]:
# Visualise: grid position vs final position regression line
pos_data = race_master.dropna(subset=['grid','positionOrder']).copy()
X_pos = pos_data[['grid']]
y_pos = pos_data['positionOrder']
X_tr2, X_ts2, y_tr2, y_ts2 = train_test_split(X_pos, y_pos, test_size=0.2, random_state=42)
lr2 = LinearRegression().fit(X_tr2, y_tr2)

x_range = np.linspace(1, 24, 100).reshape(-1,1)
plt.figure(figsize=(9,5))
plt.scatter(X_ts2, y_ts2, alpha=0.1, s=8, color='steelblue', label='Actual')
plt.plot(x_range, lr2.predict(x_range), color='red', linewidth=2.5, label='Regression Line')
plt.xlabel('Grid Position')
plt.ylabel('Final Position')
plt.title('Linear Regression: Grid → Final Position')
plt.legend()
plt.savefig('regression_grid_position.png', dpi=120)
plt.show()
print(f'R² (Grid → Position): {r2_score(y_ts2, lr2.predict(X_ts2)):.4f}')

---
## Task 8: Pit Stop Strategy Analysis

In [ ]:
# Aggregate pit stops per driver per race
pit_agg = pit_clean.groupby(['raceId','driverId']).agg(
    pit_count=('stop','count'),
    total_pit_time_sec=('duration_sec','sum'),
    avg_pit_time_sec=('duration_sec','mean')
).reset_index()

# Merge with results
pit_analysis = pit_agg.merge(
    results[['raceId','driverId','positionOrder','constructorId']],
    on=['raceId','driverId'], how='left'
).merge(
    constructors[['constructorId','name']], on='constructorId', how='left'
)
pit_analysis['positionOrder'] = pd.to_numeric(pit_analysis['positionOrder'], errors='coerce')

# Correlation: pit stop count vs final position
corr_pit = pit_analysis['pit_count'].corr(pit_analysis['positionOrder'])
print(f'Correlation (pit count vs final position): {corr_pit:.4f}')

# Scatter: Pit Stop Count vs Final Position
plt.figure(figsize=(9,5))
sns.stripplot(data=pit_analysis.dropna(subset=['positionOrder']),
              x='pit_count', y='positionOrder', alpha=0.3, jitter=True, palette='Set2')
plt.xlabel('Number of Pit Stops')
plt.ylabel('Final Finishing Position')
plt.title('Pit Stop Count vs Final Position')
plt.savefig('pit_stops_vs_position.png', dpi=120)
plt.show()

In [ ]:
# Top 10 constructors by fastest average pit stop
constructor_pit = pit_analysis.groupby('name')['avg_pit_time_sec'].mean().sort_values().head(10)

plt.figure(figsize=(11,5))
constructor_pit.plot(kind='barh', color='mediumseagreen', edgecolor='white')
plt.xlabel('Average Pit Stop Duration (seconds)')
plt.title('Top 10 Constructors – Fastest Average Pit Stop')
plt.tight_layout()
plt.savefig('fastest_pit_stop_constructors.png', dpi=120)
plt.show()

In [ ]:
# Pit stop duration trend by year
pit_year = pit_clean.merge(races[['raceId','year']], on='raceId', how='left')
pit_year_avg = pit_year.groupby('year')['duration_sec'].mean().reset_index()

plt.figure(figsize=(12,5))
plt.plot(pit_year_avg['year'], pit_year_avg['duration_sec'], marker='s', color='darkorchid', linewidth=2)
plt.xlabel('Year')
plt.ylabel('Average Pit Stop Duration (seconds)')
plt.title('Average Pit Stop Duration Trend by Year')
plt.grid(True, alpha=0.4)
plt.savefig('pit_stop_trend.png', dpi=120)
plt.show()

---
## Task 9: Overfitting and Underfitting Analysis

In [ ]:
# Polynomial degree sweep: lap number vs lap time (sample for speed)
sample_lap = lap_master[['lap','milliseconds']].dropna().sample(20000, random_state=42)
X_p = sample_lap[['lap']]
y_p = sample_lap['milliseconds']
X_p_tr, X_p_ts, y_p_tr, y_p_ts = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

degrees = range(1, 13)
train_errors, test_errors = [], []

for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d)),
        ('lr', LinearRegression())
    ])
    pipe.fit(X_p_tr, y_p_tr)
    train_errors.append(np.sqrt(mean_squared_error(y_p_tr, pipe.predict(X_p_tr))))
    test_errors.append(np.sqrt(mean_squared_error(y_p_ts, pipe.predict(X_p_ts))))

plt.figure(figsize=(10, 5))
plt.plot(degrees, train_errors, 'o-', label='Training RMSE', color='royalblue')
plt.plot(degrees, test_errors,  's-', label='Test RMSE', color='tomato')
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE (ms)')
plt.title('Overfitting/Underfitting: Training vs Test Error by Model Complexity')
plt.legend()
plt.grid(True, alpha=0.4)
plt.savefig('overfitting_curve.png', dpi=120)
plt.show()

---
## Task 10: Classification – Podium Prediction

In [ ]:
# ── Build classification dataset ─────────────────────────────────────────────
clf_df = race_master.dropna(subset=['grid','positionOrder','points']).copy()

# Target: podium = top 3
clf_df['podium'] = (clf_df['positionOrder'] <= 3).astype(int)
print(f'Class balance:\n{clf_df["podium"].value_counts(normalize=True).round(3)}')

# Encode constructor
le_c = LabelEncoder()
clf_df['constructor_enc'] = le_c.fit_transform(clf_df['name_constructor'].astype(str))

CLF_FEATURES = ['grid', 'constructor_enc', 'year']
X_clf = clf_df[CLF_FEATURES]
y_clf = clf_df['podium']

X_clf_tr, X_clf_ts, y_clf_tr, y_clf_ts = train_test_split(X_clf, y_clf, test_size=0.2,
                                                           stratify=y_clf, random_state=42)

# Scale features
scaler = StandardScaler()
X_clf_tr_s = scaler.fit_transform(X_clf_tr)
X_clf_ts_s = scaler.transform(X_clf_ts)

In [ ]:
# Logistic Regression with class balancing
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_clf_tr_s, y_clf_tr)
y_pred_clf = log_reg.predict(X_clf_ts_s)
y_prob_clf = log_reg.predict_proba(X_clf_ts_s)[:,1]

print('=== Podium Classification Report ===')
print(classification_report(y_clf_ts, y_pred_clf, target_names=['No Podium','Podium']))
print(f'ROC-AUC Score: {roc_auc_score(y_clf_ts, y_prob_clf):.4f}')

# Confusion Matrix
cm = confusion_matrix(y_clf_ts, y_pred_clf)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Podium','Podium'],
            yticklabels=['No Podium','Podium'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix – Podium Prediction')
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()

---
## Task 11 & 12: Model Evaluation – Summary Table

In [ ]:
# Regression model comparison
summary_df = pd.DataFrame(results_log)
summary_df['MAE (sec)'] = (summary_df['MAE'] / 1000).round(3)
summary_df['RMSE (sec)'] = (summary_df['RMSE'] / 1000).round(3)
summary_df['R²'] = summary_df['R2'].round(4)
display(summary_df[['Model','MAE (sec)','RMSE (sec)','R²']].style.highlight_min(
    subset=['MAE (sec)','RMSE (sec)'], color='lightgreen').highlight_max(
    subset=['R²'], color='lightgreen'))

---
## Task 13: Data Visualisation – Domain Insights

In [ ]:
# Top 10 drivers by total race wins
wins = race_master[race_master['positionOrder'] == 1].groupby('driver_name').size().sort_values(ascending=False).head(10)

plt.figure(figsize=(12,5))
wins.plot(kind='bar', color='gold', edgecolor='darkgoldenrod', linewidth=1.2)
plt.xlabel('Driver')
plt.ylabel('Race Wins')
plt.title('Top 10 F1 Drivers by Total Race Wins (All Time)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('top10_drivers_wins.png', dpi=120)
plt.show()

In [ ]:
# Constructor championship points across recent years
from sklearn.preprocessing import LabelEncoder

constructor_standings = pd.read_csv(DATA_PATH + 'constructor_standings.csv')
cs = constructor_standings.merge(races[['raceId','year']], on='raceId', how='left')
cs = cs.merge(constructors[['constructorId','name']], on='constructorId', how='left')

# Get end-of-season points for top 5 constructors
top5 = cs.groupby('name')['points'].sum().sort_values(ascending=False).head(5).index
cs_top = cs[cs['name'].isin(top5)]
cs_yearly = cs_top.groupby(['year','name'])['points'].max().reset_index()

plt.figure(figsize=(14,6))
for team in top5:
    td = cs_yearly[cs_yearly['name']==team]
    plt.plot(td['year'], td['points'], marker='o', linewidth=2, label=team, markersize=4)
plt.xlabel('Year')
plt.ylabel('Championship Points')
plt.title('Top 5 Constructor Championship Points Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('constructor_points_timeline.png', dpi=120)
plt.show()

---

## 📝 Observations and Conclusions

> **Students: Fill in your own observations below after running all cells.**

### Key Findings

1. **Lap Time Distribution:** _(Describe shape, skewness, outliers)_
2. **Grid vs Position Correlation:** _(Pearson r = ?, interpretation)_
3. **Pit Stop Strategy:** _(How does pit stop count relate to final position?)_
4. **Best Regression Model:** _(Which model gave lowest RMSE and why?)_
5. **Podium Prediction:** _(ROC-AUC, precision for podium class, key features)_
6. **Overfitting Observation:** _(At which polynomial degree did overfitting begin?)_

### Conclusion
_(Write 1 paragraph summarising what you learned from the project.)_

---
*Course 602 – Data Analytics using Python | Project by Dr. Snehal K Joshi*